In [ ]:
# Agentic AI Meeting Preparation Assistant

## Objective

Build an Agentic AI assistant that prepares a manager for an upcoming client meeting.

The assistant demonstrates:

- Retrieval-Augmented Generation (RAG)
- FAISS vector database
- Short-term conversation memory
- Long-term memory
- Agentic workflow
- Tool usage
- LLM-based meeting brief generation

## Scenario

A manager has an upcoming meeting with Acme Corp.

Instead of manually searching through client documents, previous meeting notes, and action items, the user can ask:

> Prepare me for my meeting with Acme Corp.

The agent retrieves relevant information and generates a concise meeting preparation brief.

In [1]:
!pip install numpy faiss-cpu sentence-transformers openai python-dotenv

In [2]:
#Import libraries
import os
import json
import sqlite3
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from openai import OpenAI
from datetime import datetime

In [3]:
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("OpenAI API key loaded successfully.")
else:
    print("OpenAI API key not found.")

OpenAI API key loaded successfully.


In [4]:
#Initialize OpenAI Client

client = OpenAI(api_key=api_key)
print("OpenAI client initialized.")

OpenAI client initialized.


In [8]:
#Load Documents

def load_document(filename):
    with open(filename, "r", encoding="utf-8") as file:
        return file.read()


client_document = load_document("data/ client_documents.txt")
meeting_notes = load_document("data/meeting_notes.txt")
action_items = load_document("data/action_items.txt")


print("Client document loaded:", len(client_document), "characters")
print("Meeting notes loaded:", len(meeting_notes), "characters")
print("Action items loaded:", len(action_items), "characters")

Client document loaded: 1395 characters
Meeting notes loaded: 1184 characters
Action items loaded: 733 characters


In [9]:
#Create Document Chunks

documents = [
    {
        "source": "client_documents",
        "text": client_document
    },
    {
        "source": "meeting_notes",
        "text": meeting_notes
    },
    {
        "source": "action_items",
        "text": action_items
    }
]

In [10]:
def create_chunks(text, chunk_size=500):
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks


chunks = []

for document in documents:
    document_chunks = create_chunks(document["text"])
    
    for chunk in document_chunks:
        chunks.append({
            "source": document["source"],
            "text": chunk
        })


print("Total chunks:", len(chunks))

Total chunks: 3


In [12]:
import os

SYSTEM_CA = "/etc/ssl/certs/ca-certificates.crt"

os.environ["REQUESTS_CA_BUNDLE"] = SYSTEM_CA
os.environ["SSL_CERT_FILE"] = SYSTEM_CA
os.environ["CURL_CA_BUNDLE"] = SYSTEM_CA

print("Using certificate bundle:", SYSTEM_CA)

Using certificate bundle: /etc/ssl/certs/ca-certificates.crt


In [13]:
#Load Embedding Model

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [14]:
#Generate Embeddings

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True
).astype("float32")

print("Embedding shape:", embeddings.shape)

Embedding shape: (3, 384)


In [15]:
#Normalize Embeddings

faiss.normalize_L2(embeddings)

print("Embeddings normalized.")

Embeddings normalized.


In [16]:
#Create FAISS Vector Database

dimension = embeddings.shape[1]
vector_db = faiss.IndexFlatL2(dimension)
vector_db.add(embeddings)
print("FAISS index created.")
print("Total vectors:", vector_db.ntotal)

FAISS index created.
Total vectors: 3


In [17]:
#Tool 1: Document Search

def search_documents(query, k=3):
    
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")
    
    faiss.normalize_L2(query_embedding)
    
    distances, indices = vector_db.search(
        query_embedding,
        k
    )
    
    results = []
    
    for distance, index in zip(distances[0], indices[0]):
        
        results.append({
            "source": chunks[index]["source"],
            "text": chunks[index]["text"],
            "score": float(distance)
        })
    
    return results

In [18]:
results = search_documents(
    "What are Acme Corp's main business challenges?"
)

for result in results:
    print("\nSource:", result["source"])
    print("Score:", result["score"])
    print("Text:", result["text"])


Source: action_items
Score: 0.8901423811912537
Text: ACME CORP - OPEN ACTION ITEMS Action Item 1: Owner: Analytics Team Task: Share detailed implementation timeline. Status: Open Priority: High Due Date: 29 August 2026 Action Item 2: Owner: Sales Team Task: Share commercial pricing proposal. Status: Open Priority: High Due Date: 30 August 2026 Action Item 3: Owner: Technical Team Task: Provide CRM integration documentation. Status: Open Priority: Medium Due Date: 30 August 2026 Action Item 4: Owner: Security Team Task: Share security and access control documentation. Status: Open Priority: High Due Date: 30 August 2026 Action Item 5: Owner: Analytics Team Task: Prepare customer retention dashboard demonstration. Status: Completed Priority: Medium Due Date: 27 August 2026

Source: meeting_notes
Score: 0.9101052284240723
Text: ACME CORP - PREVIOUS MEETING NOTES Meeting Date: 25 August 2026 Attendees: Sarah Johnson - VP of Operations Michael Chen - Director of Data and Analytics Mala - A

In [19]:
#Tool 2: Meeting Notes Retrieval

def retrieve_meeting_notes(client_name="Acme Corp"):
    
    query = f"""
    Previous meeting notes and discussion history for {client_name}.
    Important discussion points, client concerns, requests and decisions.
    """
    
    results = search_documents(query, k=3)
    
    meeting_results = [
        result
        for result in results
        if result["source"] == "meeting_notes"
    ]
    
    return meeting_results

In [20]:
meeting_results = retrieve_meeting_notes()

for result in meeting_results:
    print(result["text"])

ACME CORP - PREVIOUS MEETING NOTES Meeting Date: 25 August 2026 Attendees: Sarah Johnson - VP of Operations Michael Chen - Director of Data and Analytics Mala - Analytics Team Discussion: The Acme team discussed their current reporting challenges. They explained that weekly management reports are manually created using spreadsheets. Sarah was particularly interested in automated customer retention dashboards. Michael asked whether our platform can integrate with their existing CRM. The client requested information about implementation timelines and pricing. The client also asked about data security, role-based access control, and data governance. Agreed Points: 1. Our team will provide a proposed implementation timeline. 2. Our team will share pricing information. 3. Our team will provide CRM integration details. 4. Our team will share information about security and access controls. Client Feedback: The product capabilities were positively received. The main concerns are implementation

In [21]:
#Tool 3: Long-Term Memory

connection = sqlite3.connect("agent_memory.db")

cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS memories (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    client_name TEXT,
    memory TEXT,
    created_at TEXT
)
""")

connection.commit()

print("Long-term memory database ready.")

Long-term memory database ready.


In [22]:
#Store Long-Term Memory

def store_memory(client_name, memory):
    
    connection = sqlite3.connect("agent_memory.db")
    cursor = connection.cursor()
    
    cursor.execute(
        """
        INSERT INTO memories
        (client_name, memory, created_at)
        VALUES (?, ?, ?)
        """,
        (
            client_name,
            memory,
            datetime.now().isoformat()
        )
    )
    
    connection.commit()
    connection.close()
    
    print("Memory stored successfully.")
    

In [23]:
#Retrieve Long-Term Memory

def retrieve_memory(client_name):
    
    connection = sqlite3.connect("agent_memory.db")
    cursor = connection.cursor()
    
    cursor.execute(
        """
        SELECT memory, created_at
        FROM memories
        WHERE client_name = ?
        ORDER BY id DESC
        LIMIT 5
        """,
        (client_name,)
    )
    
    results = cursor.fetchall()
    
    connection.close()
    
    return results

In [24]:
#Add Some Previous Memories

store_memory(
    "Acme Corp",
    "Client is highly concerned about implementation timeline."
)

store_memory(
    "Acme Corp",
    "Sarah Johnson is particularly interested in customer retention dashboards."
)

store_memory(
    "Acme Corp",
    "Michael Chen wants detailed information about CRM integration."
)

Memory stored successfully.
Memory stored successfully.
Memory stored successfully.


In [25]:
#Test Long-Term Memory

memories = retrieve_memory("Acme Corp")
print("Long-Term Memories:\n")
for memory, created_at in memories:
    print("-", memory)
    print("  Stored:", created_at)

Long-Term Memories:

- Michael Chen wants detailed information about CRM integration.
  Stored: 2026-08-31T17:00:47.747411
- Sarah Johnson is particularly interested in customer retention dashboards.
  Stored: 2026-08-31T17:00:47.741887
- Client is highly concerned about implementation timeline.
  Stored: 2026-08-31T17:00:47.727652


In [26]:
#Short-Term Memory

short_term_memory = []

In [27]:
def add_to_short_term_memory(role, content):
    
    short_term_memory.append({
        "role": role,
        "content": content
    })

In [28]:
def get_short_term_memory():
    return short_term_memory

In [29]:
#Test Short-Term Memory

add_to_short_term_memory(
    "user",
    "Prepare me for my meeting with Acme Corp."
)

add_to_short_term_memory(
    "assistant",
    "I will gather client information, previous meeting notes and open action items."
)

print(get_short_term_memory())

[{'role': 'user', 'content': 'Prepare me for my meeting with Acme Corp.'}, {'role': 'assistant', 'content': 'I will gather client information, previous meeting notes and open action items.'}]


In [30]:
#Open Action Items Tool

def retrieve_action_items():
    
    results = search_documents(
        "Open pending action items tasks owners deadlines priorities",
        k=5
    )
    
    action_results = [
        result
        for result in results
        if result["source"] == "action_items"
    ]
    
    return action_results

In [31]:
action_results = retrieve_action_items()

for result in action_results:
    print(result["text"])

ACME CORP - OPEN ACTION ITEMS Action Item 1: Owner: Analytics Team Task: Share detailed implementation timeline. Status: Open Priority: High Due Date: 29 August 2026 Action Item 2: Owner: Sales Team Task: Share commercial pricing proposal. Status: Open Priority: High Due Date: 30 August 2026 Action Item 3: Owner: Technical Team Task: Provide CRM integration documentation. Status: Open Priority: Medium Due Date: 30 August 2026 Action Item 4: Owner: Security Team Task: Share security and access control documentation. Status: Open Priority: High Due Date: 30 August 2026 Action Item 5: Owner: Analytics Team Task: Prepare customer retention dashboard demonstration. Status: Completed Priority: Medium Due Date: 27 August 2026
ACME CORP - OPEN ACTION ITEMS Action Item 1: Owner: Analytics Team Task: Share detailed implementation timeline. Status: Open Priority: High Due Date: 29 August 2026 Action Item 2: Owner: Sales Team Task: Share commercial pricing proposal. Status: Open Priority: High Due

In [32]:
def gather_meeting_information(client_name):
    
    print("Gathering information for:", client_name)
    
    # Tool 1: RAG document search
    client_results = search_documents(
        f"{client_name} company profile business priorities challenges",
        k=3
    )
    
    # Tool 2: Previous meeting notes
    meeting_results = retrieve_meeting_notes(client_name)
    
    # Tool 3: Action items
    action_results = retrieve_action_items()
    
    # Tool 4: Long-term memory
    memory_results = retrieve_memory(client_name)
    
    return {
        "client_information": client_results,
        "meeting_notes": meeting_results,
        "action_items": action_results,
        "long_term_memory": memory_results
    }

In [33]:
#Agent Context Builder

def build_agent_context(client_name):
    
    information = gather_meeting_information(client_name)
    
    context = f"""
CLIENT: {client_name}

================ CLIENT INFORMATION ================

"""
    
    for result in information["client_information"]:
        context += result["text"] + "\n\n"
    
    context += """
================ PREVIOUS MEETING NOTES ================
"""
    
    for result in information["meeting_notes"]:
        context += result["text"] + "\n\n"
    
    context += """
================ ACTION ITEMS ================
"""
    
    for result in information["action_items"]:
        context += result["text"] + "\n\n"
    
    context += """
================ LONG-TERM MEMORY ================
"""
    
    for memory, created_at in information["long_term_memory"]:
        context += f"- {memory}\n"
    
    return context

In [34]:
#Agent Reasoning / LLM

def generate_meeting_brief(client_name):
    
    context = build_agent_context(client_name)
    
    prompt = f"""
You are an Agentic AI Meeting Preparation Assistant.

Your job is to prepare a manager for an upcoming client meeting.

You have access to information retrieved from:
1. Client documents
2. Previous meeting notes
3. Open action items
4. Long-term memory

Use the retrieved information carefully.
Do not invent facts that are not present in the retrieved context.

Prepare a concise but useful meeting brief.

Include:

1. Client Overview
2. Current Business Priorities
3. Key Challenges
4. Previous Meeting Highlights
5. Open Action Items
6. Client Concerns
7. Recommended Talking Points
8. Questions to Ask the Client
9. Suggested Next Steps

Retrieved Context:

{context}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a professional meeting preparation assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )
    
    return response.choices[0].message.content

In [35]:
#Main Agent Workflow

def meeting_preparation_agent(user_query):
    
    # ----------------------------------------
    # Step 1: Short-term memory
    # ----------------------------------------
    
    add_to_short_term_memory(
        "user",
        user_query
    )
    
    print("\n[Agent] Understanding user request...")
    
    # ----------------------------------------
    # Step 2: Identify client
    # ----------------------------------------
    
    client_name = "Acme Corp"
    
    print("[Agent] Identified client:", client_name)
    
    # ----------------------------------------
    # Step 3: Use tools
    # ----------------------------------------
    
    print("[Agent] Searching client documents...")
    
    client_results = search_documents(
        f"{client_name} company profile priorities challenges",
        k=3
    )
    
    print("[Agent] Retrieving previous meeting notes...")
    
    meeting_results = retrieve_meeting_notes(client_name)
    
    print("[Agent] Retrieving open action items...")
    
    action_results = retrieve_action_items()
    
    print("[Agent] Retrieving long-term memory...")
    
    memory_results = retrieve_memory(client_name)
    
    # ----------------------------------------
    # Step 4: Generate meeting brief
    # ----------------------------------------
    
    brief = generate_meeting_brief(client_name)
    
    # ----------------------------------------
    # Step 5: Store result in short-term memory
    # ----------------------------------------
    
    add_to_short_term_memory(
        "assistant",
        brief
    )
    
    # ----------------------------------------
    # Step 6: Store useful long-term memory
    # ----------------------------------------
    
    store_memory(
        client_name,
        f"Meeting preparation requested on {datetime.now().strftime('%Y-%m-%d')}"
    )
    
    return brief

In [ ]:
#Run the Agent

user_query = "Prepare me for my meeting with Acme Corp."

meeting_brief = meeting_preparation_agent(user_query)

print("\n")
print("=" * 100)
print("MEETING PREPARATION BRIEF")
print("=" * 100)

print(meeting_brief)   

In [ ]:
#Follow-up Question

follow_up = """
What are the most important questions I should ask Sarah Johnson
during the Acme Corp meeting?
"""

add_to_short_term_memory(
    "user",
    follow_up
)

print("Conversation History:")
print("-" * 80)

for message in get_short_term_memory():
    print(message["role"], ":", message["content"][:300])

In [37]:
#Demonstrate Long-Term Memory

print("Long-Term Memory for Acme Corp")
print("=" * 80)

memories = retrieve_memory("Acme Corp")

for memory, timestamp in memories:
    print("-", memory)
    print(" ", timestamp)

Long-Term Memory for Acme Corp
- Michael Chen wants detailed information about CRM integration.
  2026-08-31T17:00:47.747411
- Sarah Johnson is particularly interested in customer retention dashboards.
  2026-08-31T17:00:47.741887
- Client is highly concerned about implementation timeline.
  2026-08-31T17:00:47.727652


In [ ]:
print("Long-Term Memory for Acme Corp")
print("=" * 80)

memories = retrieve_memory("Acme Corp")

for memory, timestamp in memories:
    print("-", memory)
    print(" ", timestamp)